# LAB | Connecting Python to SQL

In [1]:
# STEP 1 - Import libraries
import pandas as pd
from sqlalchemy import create_engine

In [6]:
# STEP 2 - Connect to MySQL
user = "root"
password = "SQLouraley1235"
host = "localhost"
port = 3306
database = "sakila"

engine = create_engine(
    f"mysql+mysqlconnector://{user}:{password}@{host}:{port}/{database}"
)

print("Connection engine created successfully.")

Connection engine created successfully.


In [7]:
# TEST THE DATABASE CONNECTION

test_query = """
SELECT *
FROM rental
LIMIT 5;
"""

pd.read_sql(test_query, engine)

,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1,2005-05-24 22:53:30,367,130,2005-05-26 22:04:30,1,2006-02-15 21:30:53
1,2,2005-05-24 22:54:33,1525,459,2005-05-28 19:40:33,1,2006-02-15 21:30:53
2,3,2005-05-24 23:03:39,1711,408,2005-06-01 22:12:39,1,2006-02-15 21:30:53
3,4,2005-05-24 23:04:41,2452,333,2005-06-03 01:43:41,2,2006-02-15 21:30:53
4,5,2005-05-24 23:05:21,2079,222,2005-06-02 04:33:21,1,2006-02-15 21:30:53


In [8]:
# STEP 3 - rentals_month()

def rentals_month(engine, month, year):
    """
    Retrieve rental data for a given month and year.
    """

    query = f"""
    SELECT *
    FROM rental
    WHERE MONTH(rental_date) = {month}
      AND YEAR(rental_date) = {year};
    """

    rentals = pd.read_sql(query, engine)

    return rentals

In [9]:
may_2005 = rentals_month(engine, 5, 2005)

may_2005.head()

,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1,2005-05-24 22:53:30,367,130,2005-05-26 22:04:30,1,2006-02-15 21:30:53
1,2,2005-05-24 22:54:33,1525,459,2005-05-28 19:40:33,1,2006-02-15 21:30:53
2,3,2005-05-24 23:03:39,1711,408,2005-06-01 22:12:39,1,2006-02-15 21:30:53
3,4,2005-05-24 23:04:41,2452,333,2005-06-03 01:43:41,2,2006-02-15 21:30:53
4,5,2005-05-24 23:05:21,2079,222,2005-06-02 04:33:21,1,2006-02-15 21:30:53


In [10]:
# STEP 4 - rental_count_month()

def rental_count_month(df, month, year):
    """
    Count rentals made by each customer for a given month and year.
    """

    column_name = f"rentals_{month:02d}_{year}"

    rental_counts = (
        df.groupby("customer_id")
          .size()
          .reset_index(name=column_name)
    )

    return rental_counts

In [11]:
may_count = rental_count_month(may_2005, 5, 2005)

may_count.head()

,customer_id,rentals_05_2005
0,1,2
1,2,1
2,3,2
3,5,3
4,6,3


In [12]:
# STEP 5 - compare_rentals()

def compare_rentals(df1, df2):
    """
    Compare the number of rentals made by each customer
    in two different months.
    """

    comparison = pd.merge(
        df1,
        df2,
        on="customer_id",
        how="outer"
    )

    comparison = comparison.fillna(0)

    comparison["difference"] = (
        comparison.iloc[:, 2] - comparison.iloc[:, 1]
    )

    return comparison

In [13]:
# STEP 6 - Test the functions

# Rentals for May
may_2005 = rentals_month(engine, 5, 2005)

# Rentals for June
june_2005 = rentals_month(engine, 6, 2005)

# Count rentals
may_count = rental_count_month(may_2005, 5, 2005)
june_count = rental_count_month(june_2005, 6, 2005)

# Compare
comparison = compare_rentals(may_count, june_count)

comparison.head()

,customer_id,rentals_05_2005,rentals_06_2005,difference
0,1,2.0,7.0,5.0
1,2,1.0,1.0,0.0
2,3,2.0,4.0,2.0
3,4,0.0,6.0,6.0
4,5,3.0,5.0,2.0


In [14]:
comparison.info()

<class 'pandas.DataFrame'>
RangeIndex: 598 entries, 0 to 597
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   customer_id      598 non-null    int64  
 1   rentals_05_2005  598 non-null    float64
 2   rentals_06_2005  598 non-null    float64
 3   difference       598 non-null    float64
dtypes: float64(3), int64(1)
memory usage: 18.8 KB


In [15]:
comparison.head(10)

,customer_id,rentals_05_2005,rentals_06_2005,difference
0,1,2.0,7.0,5.0
1,2,1.0,1.0,0.0
2,3,2.0,4.0,2.0
3,4,0.0,6.0,6.0
4,5,3.0,5.0,2.0
5,6,3.0,4.0,1.0
6,7,5.0,5.0,0.0
7,8,1.0,3.0,2.0
8,9,3.0,2.0,-1.0
9,10,1.0,5.0,4.0
